# Global inhibition: Subtractive vs Divisive

Toggle between models to compare their behavior with the same parameters.

- **Subtractive**: $\dot{x}_i + x_i = b_i - [y]_+^r$, $\quad \dot{y} + y = \sum_j [x_j]_+^q$
- **Divisive**: $\dot{x}_i + x_i\,(1 + [y]_+^r) = b_i$, $\quad \dot{y} + y = \sum_j [x_j]_+^q$

In [6]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Button, HBox, VBox, interactive_output
from IPython.display import display

def relu_pow(x, p):
    return np.maximum(x, 0) ** p

def pow(x, p):
    return x ** p

In [ ]:
from ipywidgets import Dropdown

state_cmp = {'b': None, 'randomized': False}

def simulate_cmp(model='Subtractive', n=10, bmax=1.0, q=2.0, r=1.0, beta=1.0, dt=0.1, steps=100, _trigger=0):
    if state_cmp['randomized'] and state_cmp['b'] is not None and len(state_cmp['b']) == n:
        b = state_cmp['b']
    else:
        b = np.linspace(bmax / n, bmax, n)
        state_cmp['b'] = b
        state_cmp['randomized'] = False

    f = lambda x: relu_pow(x, q)

    x = np.zeros(n)
    y = 0.0
    y_d = 0.0

    X = np.zeros((steps + 1, n))
    Y = np.zeros(steps + 1)
    Y_d = np.zeros(steps + 1)
    X[0] = x
    Y[0] = y
    Y_d[0] = y_d

    if model == 'Subtractive':
        for t in range(steps):
            x += dt * (b - x - relu_pow(y, r))
            y += dt * (np.sum(f(x)) - y)
            X[t + 1] = x
            Y[t + 1] = y
    elif model == 'Divisive':
        g = lambda yy: 1.0 + relu_pow(yy, r)
        for t in range(steps):
            x += dt * (b - x * g(y))
            y += dt * (np.sum(f(x)) - y)
            X[t + 1] = x
            Y[t + 1] = y
    else:
        # Combined model: subtractive term and divisive gain control together.
        # x_i. + x_i * (1 + y_d^r) = b_i - beta * y
        # y. + y = 1/n * sum(x_i)
        # y_d. + y_d = 1/n * sum(x_i^2)
        g = lambda yy: 1.0 + pow(yy, r)
        for t in range(steps):
            x += dt * (b - x * g(y_d) - beta * y)
            y += dt * (np.sum(x) / n - y)
            y_d += dt * (np.sum(x ** 2) / n - y_d)
            X[t + 1] = x
            Y[t + 1] = y
            Y_d[t + 1] = y_d

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(8, 6))
    time = np.arange(steps + 1) * dt
    idx = np.arange(1, n + 1)
    if model == 'Subtractive':
        color = 'C0'
    elif model == 'Divisive':
        color = 'C2'
    else:
        color = 'C1'

    # Trajectories of x_i
    ax = axes[0, 0]
    for i in range(n):
        ax.plot(time, X[:, i], color=plt.cm.viridis(i / n))
    ax.set_xlabel('time')
    ax.set_ylabel('$x_i$')
    ax.set_title(f'Excitatory preactivities ({model})')

    # Trajectory of y
    ax = axes[0, 1]
    ax.plot(time, Y, color='red')
    if model == 'Combined':
        ax.plot(time, Y_d, color='orange', linestyle='--', label='$y_d$')
        ax.legend(loc='best')
    ax.set_xlabel('time')
    ax.set_ylabel('$y$')
    ax.set_title('Inhibitory preactivity')

    # Steady-state output
    ax = axes[1, 0]
    ax.bar(idx, f(X[-1]), color=color, alpha=0.7)
    ax.set_xlabel('$i$')
    ax.set_ylabel('$f(x_i)$', color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.set_title('Steady-state activity')
    ax2 = ax.twinx()
    ax2.plot(idx, b, 'o-', color='gray')
    ax2.set_ylabel('$b_i$', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')

    # Steady-state preactivity
    ax = axes[1, 1]
    ax.bar(idx, X[-1], color=color, alpha=0.7)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_xlabel('$i$')
    ax.set_ylabel('$x_i$', color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.set_title('Steady-state preactivity')
    ax2 = ax.twinx()
    ax2.plot(idx, b, 'o-', color='gray')
    ax2.set_ylabel('$b_i$', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    # Report mean and std of x_i in the title
    mean_x = np.mean(X[-1])
    std_x = np.std(X[-1], ddof=1)
    ax.set_title(f'Steady-state preactivity\n(mean={mean_x:.2f}, std={std_x:.2f})')
    # If it is indeed z-scored, we should see mean close to 0 and std close to 1.

    plt.tight_layout()
    plt.show()

# Sliders
model_dropdown = Dropdown(options=['Subtractive', 'Divisive', 'Combined'], value='Subtractive', description='Model')
n_slider_c = IntSlider(min=2, max=50, step=1, value=10, description='n')
bmax_slider_c = FloatSlider(min=0.1, max=20, step=0.1, value=1.0, description='b_max')
q_slider_c = FloatSlider(min=0.5, max=5.0, step=0.1, value=2.0, description='q')
r_slider_c = FloatSlider(min=0.5, max=100.0, step=0.1, value=1.0, description='r')
beta_slider_c = FloatSlider(min=0.0, max=100.0, step=0.1, value=1.0, description='beta')
dt_slider_c = FloatSlider(min=0.1, max=1, step=0.01, value=0.1, description='dt')
steps_slider_c = IntSlider(min=100, max=10000, step=100, value=100, description='steps')
trigger_c = IntSlider(value=0)

randomize_btn_c = Button(description='Randomize b')
reset_btn_c = Button(description='Reset b')

def on_randomize_c(btn):
    n = n_slider_c.value
    bmax = bmax_slider_c.value
    state_cmp['b'] = np.random.uniform(0, bmax, n)
    state_cmp['randomized'] = True
    trigger_c.value += 1

def on_reset_c(btn):
    state_cmp['randomized'] = False
    state_cmp['b'] = None
    trigger_c.value += 1

randomize_btn_c.on_click(on_randomize_c)
reset_btn_c.on_click(on_reset_c)

def on_param_change_c(change):
    state_cmp['randomized'] = False
    state_cmp['b'] = None

n_slider_c.observe(on_param_change_c, names='value')
bmax_slider_c.observe(on_param_change_c, names='value')
beta_slider_c.observe(on_param_change_c, names='value')

ui_c = VBox([
    HBox([model_dropdown]),
    HBox([n_slider_c, bmax_slider_c]),
    HBox([q_slider_c, r_slider_c]),
    HBox([beta_slider_c]),
    HBox([dt_slider_c, steps_slider_c]),
    HBox([randomize_btn_c, reset_btn_c])
])

out_c = interactive_output(simulate_cmp, {
    'model': model_dropdown,
    'n': n_slider_c, 'bmax': bmax_slider_c,
    'q': q_slider_c, 'r': r_slider_c, 'beta': beta_slider_c,
    'dt': dt_slider_c, 'steps': steps_slider_c,
    '_trigger': trigger_c
})

display(ui_c, out_c)

Output()